# 04 · Third places

Where do people work when they work neither at home nor at the office?
This notebook follows the paper's three complementary routes
(`notebook_docs/methodology/THIRD_PLACES.md`); parameters are in `configs/third_places.json`,
with small-sample overrides for the demo in `configs/third_places_demo.json`.

1. **Visits**: dwells matched to the nearest POI within `poi_matching.radius_m`, with
   context — distance to home / work, what the device did just before, and the device's
   work schedule (remote / onsite) from the remote-work metric.
2. **Rules**: POI visit statistics (weekday, late-morning, long stays) and flags by
   category, location and the behaviour of remote workers; z-score **candidates**.
3. **PU model**: places with remote-work evidence in reviews are *positives*, everything
   else is *unlabelled*; an Elkan–Noto classifier ranks all places.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely

from remote_work_utils import third_places as tp
from remote_work_utils.pipeline import output_path
from remote_work_utils.plotting import plot_ci_dots, share_by
from remote_work_utils.schema import read_dwells
from remote_work_utils.settings import load_settings, resolve_path

settings = load_settings("../configs/demo.json")
config = load_settings("../configs/third_places_demo.json")
months = settings["aggregation"]["months"]

places = pd.read_csv(resolve_path(settings, "places"))
pois = gpd.GeoDataFrame(places.rename(columns={"place_id": "poi_id"}),
                        geometry=gpd.points_from_xy(places["lon"], places["lat"]), crs=4326)
stat_areas = gpd.read_file(resolve_path(settings, "stat_areas"))

## 1. Visits
Uses the outputs of notebook 03 (home / work locations and the user remote-work table)
and writes one visits table per month for notebooks 05 and 06.

In [ ]:
frames = []
for month in months:
    visits = tp.visits_for_month(
        read_dwells(resolve_path(settings, "dwells", month=month), settings),
        pois,
        pd.read_csv(output_path(settings, "work", month)),
        pd.read_csv(output_path(settings, "remote_work_users", month)),
        stat_areas,
        settings,
        config,
    ).assign(month=month)
    points = visits["the_geom"].to_numpy()
    visits.drop(columns="the_geom").assign(lon=shapely.get_x(points), lat=shapely.get_y(points)).to_parquet(
        output_path(settings, "visits", month), index=False
    )
    frames.append(visits)
visits = pd.concat(frames, ignore_index=True)
print(f"{len(visits):,} dwells, {visits['poi_id'].notna().mean():.1%} at a POI")
visits.groupby("month")["work_schedule"].value_counts().unstack()

## How visits differ by category
Metric: long (> 1 h) weekday visit starting 10:00–14:00 · Categories: POI category ·
X: share of visits (%) with 95% Wilson CI. Unit: visits outside home and work.

In [ ]:
visit_stats = config["visit_stats"]
at_poi = visits[visits["poi_id"].notna() & ~visits["flag_home"] & ~visits["flag_work"]].copy()
at_poi = at_poi.merge(places.rename(columns={"place_id": "poi_id"})[["poi_id", "category"]], on="poi_id")
start_hour = at_poi["start_date_time"].dt.hour
weekday = pd.to_datetime(at_poi["date"]).dt.weekday.isin(visit_stats["day_types"]["weekday"])
window = next(w for w in visit_stats["time_periods"] if w["name"] in visit_stats["remote_work_time_periods"])
in_window = (start_hour >= window["start_hour"]) & (start_hour < window["end_hour"])
at_poi["long_late_morning_weekday"] = weekday & in_window & (at_poi["duration_hours"] > visit_stats["long_visit_hours"])

table = share_by(at_poi, ["category"], "long_late_morning_weekday").sort_values("share", ascending=False)
fig, ax = plt.subplots(figsize=(6, 4))
plot_ci_dots(ax, table, "category", "share", overall=at_poi["long_late_morning_weekday"].mean(), scale=100)
ax.set_title("Long late-morning weekday visits by POI category")
ax.set_xlabel("Share of visits (%)")
plt.tight_layout()
plt.show()

> **Fig. 4.1.** Share of visits that are long weekday visits starting between 10:00 and 14:00, by
> POI category. Cafes, coworking spaces and libraries stand out. *Notes:* synthetic data,
> three months; unit = visits outside home and work, so frequent visitors weigh more;
> 95% Wilson CIs.

## 2. Rules: POI statistics, flags and candidates

In [ ]:
outside = visits[~visits["flag_home"] & ~visits["flag_work"]]
stats = tp.poi_visit_stats(outside, visit_stats).merge(
    places.rename(columns={"place_id": "poi_id"}), on="poi_id"
)
remote = tp.poi_visit_stats(outside[outside["work_schedule"] == "remote_worker"], visit_stats)
remote["flag_user_behaviour_suits_rw"] = tp.flag_user_behaviour(remote, config["poi_flags"])
stats = stats.merge(remote[["poi_id", "rw_share", "flag_user_behaviour_suits_rw"]].rename(
    columns={"rw_share": "rw_share_remote_workers"}), on="poi_id", how="left")
stats["flag_category_suits_rw"] = tp.flag_category(stats["category"], config["poi_flags"])
stats[["poi_id", "category", "n_visits", "share_weekday", "share_early_afternoon",
       "avg_duration_hours_weekday", "rw_share_remote_workers", "flag_user_behaviour_suits_rw",
       "flag_category_suits_rw"]].sort_values("rw_share_remote_workers", ascending=False).head(10)

The category flag uses Google Places types: `coworking_space` counts as an *office*
(not a third place), and the synthetic label `coworking` is unknown to it (missing).
The location flag (`flag_location`) needs land-use counts per geohash, which the synthetic
city does not have.

In [ ]:
candidates = tp.find_candidates(tp.scores_from_visit_stats(stats, config["candidates"]), config["candidates"])
pd.crosstab(candidates["place_group"], candidates["is_candidate_all"], margins=True)

## 3. PU model
Reviews are not synthetic, so we simulate the review evidence: half of the work-friendly
places are labelled positive, all other places are unlabelled. The model never sees
`place_group`.

In [ ]:
pu = config["pu_model"]
features = ["avg_duration_hours_weekday", "share_weekday", "share_early_afternoon", "share_evening",
            "share_saturday", "avg_dist_home", "visit_frequency", "share_visits_after_home"]
X = tp.impute(stats[features], pu["imputation"])
work_friendly = (stats["place_group"] == "work_friendly").to_numpy()
rng = np.random.default_rng(pu["random_state"])
labelled = (work_friendly & (rng.random(len(stats)) < 0.5)).astype(int)
print(f"{labelled.sum()} labelled positives among {len(stats)} places ({work_friendly.sum()} truly work-friendly)")

model = tp.make_model(pu["C"], pu["l1_ratio"], pu["max_iter"], pu["random_state"]).fit(X, labelled)
stats["p_remote_work_place"] = model.predict_proba(X)
tp.coefficients(model, features)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for group, color in [("work_friendly", "#2f855a"), ("leisure", "#dd6b20")]:
    ax.hist(stats.loc[stats["place_group"] == group, "p_remote_work_place"], bins=np.linspace(0, 1, 21),
            alpha=0.7, color=color, label=group.replace("_", "-"))
ax.set_title("PU model scores by true place type")
ax.set_xlabel("P(remote-work place | features)")
ax.set_ylabel("Places (count)")
ax.legend()
plt.show()

> **Fig. 4.2.** Elkan–Noto PU scores of all places, by true (synthetic) type. Unlabelled
> work-friendly places score as high as the labelled ones. *Notes:* elastic-net logistic
> regression on standardised visit features; C and l1 ratio from `pu_model`.

### Validation without negatives
Hide a share of the positives, retrain, and check how high the hidden ones rank.

In [ ]:
holdout = tp.positive_holdout_validation(X, labelled, pu, pu["C"], pu["l1_ratio"])
holdout.drop(columns="repeat").mean().round(3)

In [ ]:
enrichment = tp.hidden_positive_enrichment(X, labelled, pu)
curve = enrichment.groupby(["C", "top_share"])["share_of_hidden_positives_recovered"].mean().unstack(0)

fig, ax = plt.subplots(figsize=(6, 4))
for C in curve.columns:
    ax.plot(curve.index * 100, curve[C] * 100, label=f"C = {C}")
ax.plot([0, 100], [0, 100], "k--", lw=1, label="Random ranking")
ax.set_title("Hidden positives recovered in the top-ranked places")
ax.set_xlabel("Top places by score (%)")
ax.set_ylabel("Hidden positives recovered (%)")
ax.legend()
plt.show()

> **Fig. 4.3.** Enrichment curves: share of hidden positive places recovered among the top x% of
> places, averaged over random holdouts, for several regularisation strengths. *Notes:* 30% of
> positives hidden per repeat; synthetic data.